# Lab 13 — Directional Feature Stress & Model Sensitivity

*AI Cybersecurity Masterclass · Lab 13 · AI60secLab*

**Companion video:** none yet for this lab.

**Research question:** Which telemetry features produce the largest change in attack detection when their observed values move within training-data bounds?

This is an offline defensive sensitivity experiment. We change one feature at a time on copied attack rows and measure the model response.

In [ ]:
# Setup: works in Google Colab and locally. No uploads needed.
# In Colab this clones the course repo, which already includes the UNSW-NB15 data.
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = "ai-cybersecurity-public-labs"

try:
    import google.colab  # noqa: F401
    if not Path(REPO_NAME).exists():
        !git clone --depth 1 {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from src.data_loader import load_unsw
from src.preprocessing import split_xy, align_columns
from src.features import drop_identifier_like_columns
from src.models import random_forest_pipeline
from src.robustness import feature_bounds, feature_stress_curve

train, test = load_unsw(DATA_DIR)

X_train, y_train = split_xy(train, "label")
X_test, y_test = split_xy(test, "label")
X_train, X_test = align_columns(X_train, X_test)

X_train = drop_identifier_like_columns(X_train)
X_test = drop_identifier_like_columns(X_test)

model = random_forest_pipeline(X_train)
model.fit(X_train, y_train)

X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

attack_mask = y_test == 1
X_attack = X_test.loc[attack_mask].reset_index(drop=True)
y_attack = y_test.loc[attack_mask].reset_index(drop=True)

In [ ]:
candidate_features = [
    c for c in [
        "dur", "sbytes", "dbytes", "spkts", "dpkts",
        "rate", "sload", "dload", "sinpkt", "dinpkt"
    ]
    if c in X_train.columns
]

bounds = feature_bounds(X_train, candidate_features)
print("Stress features:", candidate_features)

In [ ]:
multipliers = [0.50, 0.75, 1.00, 1.25, 1.50]
curves = []

for feature in candidate_features[:8]:
    curves.append(
        feature_stress_curve(
            model,
            X_attack,
            y_attack,
            feature=feature,
            multipliers=multipliers,
            bounds=bounds,
        )
    )

sensitivity = pd.concat(curves, ignore_index=True)
display(sensitivity.head(20).round(4))

In [ ]:
baseline = (
    sensitivity[sensitivity["multiplier"] == 1.0]
    [["feature", "recall"]]
    .rename(columns={"recall": "baseline_recall"})
)

worst = (
    sensitivity.groupby("feature", as_index=False)
    .agg(min_recall=("recall", "min"))
    .merge(baseline, on="feature", how="left")
)
worst["recall_drop"] = (
    worst["baseline_recall"] - worst["min_recall"]
)

display(
    worst.sort_values("recall_drop", ascending=False).round(4)
)

In [ ]:
top_feature = (
    worst.sort_values("recall_drop", ascending=False)
    .iloc[0]["feature"]
)

plot_data = sensitivity[
    sensitivity["feature"] == top_feature
]

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(
    plot_data["multiplier"],
    plot_data["recall"],
    marker="o",
)
ax.set_xlabel("Feature multiplier")
ax.set_ylabel("Attack recall")
ax.set_title(f"Sensitivity curve — {top_feature}")
ax.set_ylim(0, 1.02)
plt.show()

## Interpretation

Explain which feature produced the largest recall degradation, whether the effect was symmetric, and why one-at-a-time stress does not prove that an attacker can reliably evade the detector.

**Takeaway:** sensitivity analysis identifies fragile model dependencies that deserve deeper robustness testing.